# Phase 1 – Step 1.2: Data Exploration

**Purpose:** understand the three CSVs *before* writing any schema. Every validation rule in Step 1.3
should trace back to something we observe here.

**What you will learn**
1. How the files relate (customer → accounts → transactions)
2. Which columns are categorical — these become Pydantic `Literal[...]` types
3. Which data-quality quirks the schemas must *tolerate* (flag, not reject)
4. What suspicious activity looks like in numbers — and how the data accidentally leaks the answer

> Run cells top to bottom. Kernel: the project's `.venv` (Python 3.11).

## 0. Setup
The notebook runs from `notebooks/`, so we add the repo root to `sys.path` to import `src.config`.

In [ ]:
# Phase 1 – Step 1.2: make `src` importable and load the CSVs as plain strings
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.config import DATA_DIR

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)


def load(name: str) -> pd.DataFrame:
    # dtype=str: nothing is silently converted; keep_default_na=False: blanks stay "" (not NaN)
    return pd.read_csv(DATA_DIR / f"{name}.csv", dtype=str, keep_default_na=False)


customers, accounts, transactions = load("customers"), load("accounts"), load("transactions")
{name: df.shape for name, df in
 [("customers", customers), ("accounts", accounts), ("transactions", transactions)]}

## 1. First look at each file
**Why `dtype=str`?** pandas would otherwise guess types: `ssn_last_4 = "0123"` would become the
integer `123` and lose its leading zero. We load *everything* as text and let Pydantic do the typing
and validation in Step 1.3.

In [ ]:
customers.head(3)

In [ ]:
accounts.head(3)

In [ ]:
transactions.head(3)

### Blank values per column
CSV has no real *null* — only empty strings. These columns will become `Optional` (blank → `None`).

In [ ]:
# Phase 1 – Step 1.2: count empty strings per column in each file
pd.DataFrame({
    name: (df == "").sum()
    for name, df in [("customers", customers), ("accounts", accounts), ("transactions", transactions)]
}).fillna("-")

## 2. How the files join
Expected: every account points to a real customer, every transaction to a real account.
**Orphans** (rows pointing at something that doesn't exist) would break `CaseData`.

In [ ]:
# Phase 1 – Step 1.2: referential-integrity checks
cust_ids, acct_ids = set(customers.customer_id), set(accounts.account_id)
print("orphan accounts     :", (~accounts.customer_id.isin(cust_ids)).sum())
print("orphan transactions :", (~transactions.account_id.isin(acct_ids)).sum())
print("accounts w/o txns   :", len(acct_ids - set(transactions.account_id)))
print("duplicate txn ids   :", transactions.transaction_id.duplicated().sum())
print("account_id starts with its customer_id:",
      all(a.startswith(c + "_") for a, c in zip(accounts.account_id, accounts.customer_id)))

In [ ]:
# Phase 1 – Step 1.2: how many accounts does a customer hold?
accounts.groupby("customer_id").size().value_counts().rename_axis("accounts_per_customer").to_frame("customers")

## 3. Categorical columns → `Literal[...]`
A small, fixed set of values is a *category*. In Step 1.3 each becomes a `Literal`, so a typo such
as `"Hgih"` fails validation loudly instead of quietly creating a new category.

In [ ]:
# Phase 1 – Step 1.2: value sets for every categorical column
categoricals = {
    "customers.risk_rating": customers.risk_rating,
    "accounts.account_type": accounts.account_type,
    "accounts.status": accounts.status,
    "transactions.transaction_type": transactions.transaction_type,
    "transactions.method": transactions.method,
    "transactions.location": transactions.location,
}
for name, col in categoricals.items():
    print(f"{name}  ({col.nunique()} values)")
    print("   ", col.value_counts().to_dict(), "\n")

## 4. Numeric and date ranges
These ranges justify rules like *amount > 0* and *balance ≥ 0*.

In [ ]:
# Phase 1 – Step 1.2: convert copies to numbers/dates just for inspection
num = pd.DataFrame({
    "annual_income": pd.to_numeric(customers.annual_income).describe(),
    "current_balance": pd.to_numeric(accounts.current_balance).describe(),
    "avg_monthly_balance": pd.to_numeric(accounts.average_monthly_balance).describe(),
    "txn_amount": pd.to_numeric(transactions.amount).describe(),
})
num.round(2)

In [ ]:
# Phase 1 – Step 1.2: date ranges — anything in the future would be invalid
for label, col in [("date_of_birth", customers.date_of_birth),
                   ("customer_since", customers.customer_since),
                   ("opening_date", accounts.opening_date),
                   ("transaction_date", transactions.transaction_date)]:
    d = pd.to_datetime(col, format="%Y-%m-%d")
    print(f"{label:17s} {d.min().date()} to {d.max().date()}")

In [ ]:
# Phase 1 – Step 1.2: ssn_last_4 must be exactly 4 digits (kept as text)
print("all 4-digit strings:", customers.ssn_last_4.str.fullmatch(r"\d{4}").all())
print("phone formats are inconsistent, e.g.:", customers.phone.sample(5, random_state=1).tolist())

## 5. Data-quality quirks (flag, don't reject)
Decision from the Stage 01 review: a bank cannot drop transactions it dislikes, so these become
`data_quality_flags` on the case instead of validation errors.

In [ ]:
# Phase 1 – Step 1.2: transactions dated before their account was opened
tx = transactions.merge(accounts[["account_id", "opening_date"]], on="account_id")
before_open = tx[tx.transaction_date < tx.opening_date]   # ISO dates compare correctly as strings
print("transactions before account opening:", len(before_open))
before_open[["transaction_id", "account_id", "transaction_date", "opening_date"]].head()

In [ ]:
# Phase 1 – Step 1.2: transaction_type vs method — e.g. an 'Online_Transfer' done at an 'ATM'
pd.crosstab(transactions.transaction_type, transactions.method)

In [ ]:
# Phase 1 – Step 1.2: zero average balance — is it the accounts that have no transactions?
zero_avg = set(accounts.loc[accounts.average_monthly_balance == "0.0", "account_id"])
no_txn = acct_ids - set(transactions.account_id)
print("avg balance 0.0:", len(zero_avg), "| no transactions:", len(no_txn),
      "| same accounts:", zero_avg == no_txn)

## 6. What does suspicious activity look like?
Common red flags a Risk Analyst looks for:

| Red flag | Why it matters | Likely typology |
|---|---|---|
| Cash deposits just under **$10,000** | Banks must file a Currency Transaction Report at > $10k; splitting deposits to avoid it is **structuring** (31 U.S.C. 5324) | Structuring |
| Very large cash transactions | Hard to trace source of funds | Money laundering |
| Large wires in and quickly out | "Layering" — moving money to obscure its origin | Money laundering |
| Activity across many branches | Spreading activity to avoid notice | Structuring / Other |
| Bursts of many small transfers | Testing accounts, mule activity | Fraud / Other |

In [ ]:
# Phase 1 – Step 1.2: cash deposits between $9,000 and $9,999.99, grouped by account
t = transactions.assign(amount=pd.to_numeric(transactions.amount))
near_ctr = t[(t.transaction_type == "Cash_Deposit") & t.amount.between(9000, 9999.99)]
near_ctr.groupby("account_id").agg(deposits=("amount", "size"), total=("amount", "sum"),
                                   first=("transaction_date", "min"),
                                   last=("transaction_date", "max")).sort_values("deposits",
                                                                                 ascending=False)

In [ ]:
# Phase 1 – Step 1.2: transactions of $10,000 or more by type
t[t.amount >= 10_000].groupby("transaction_type").amount.agg(["size", "min", "max"]).round(2)

In [ ]:
# Phase 1 – Step 1.2: accounts using the most distinct branches
t[t.location != ""].groupby("account_id").location.nunique().sort_values(ascending=False).head(8)

## 7. ⚠️ The transaction ID gives the answer away (data leakage)
Look at the **prefix** of `transaction_id`. Most are `TXN_`, but some are named after the pattern
that was planted in the synthetic data.

In [ ]:
# Phase 1 – Step 1.2: transaction counts and characteristics by ID prefix
t["prefix"] = t.transaction_id.str.split("_").str[0]
t.groupby("prefix").agg(txns=("amount", "size"), accounts=("account_id", "nunique"),
                        min_amt=("amount", "min"), max_amt=("amount", "max"),
                        top_type=("transaction_type", lambda s: s.mode().iat[0])).round(2)

In [ ]:
# Phase 1 – Step 1.2: our 'near $10k cash' rule vs the STRUCT_ label — do they agree?
pd.crosstab(near_ctr.transaction_id.str.split("_").str[0], "count")

**What this means for the project**
- If the LLM sees `STRUCT_7D3B8E94` it can classify by *reading the label* instead of reasoning.
  That's **data leakage** — results would look great and mean nothing.
- **Stage 01:** keep `transaction_id` unchanged — it is the record's identity for the audit trail.
- **Stage 02:** the prompt formatter **masks** IDs before they reach the model.
- **Stage 08:** the prefixes are free **ground-truth labels** to measure the agent's accuracy.

In [ ]:
# Phase 1 – Step 1.2: which customers carry planted patterns? (future eval labels)
labelled = t[t.prefix != "TXN"].merge(accounts[["account_id", "customer_id"]], on="account_id")
labelled.groupby("customer_id").prefix.agg(lambda s: sorted(set(s))).to_frame("patterns")

## 8. Findings → Step 1.3 schema rules

| Finding | Schema rule |
|---|---|
| IDs follow `CUST_####`, `CUST_####_ACC_n`, `PREFIX_XXXXXXXX` | regex `pattern` on ID fields |
| `account_id` always starts with its `customer_id` | `model_validator` on `AccountData` |
| `risk_rating`, `account_type`, `status`, `transaction_type`, `method` are small fixed sets | `Literal[...]` |
| `ssn_last_4` is 4 digits with possible leading zeros | `str` with pattern `^\d{4}$` |
| Amounts all > 0, balances all ≥ 0 | `gt=0` / `ge=0` |
| `counterparty`, `location` often blank | `str \| None`, blank → `None` |
| Phone formats vary | keep as free text |
| Transactions before account opening; type/method mismatch | `data_quality_flags`, not errors |
| 37 accounts have no transactions | `CaseData.transactions` may be empty |
| ID prefixes reveal planted patterns | keep IDs; mask in Stage 02 prompts |